# 🔁 Notebook 4 — Bidirectional LSTM Model

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from src.models.lstm_model import LSTMClassifier
from config import EPOCHS, BATCH_SIZE, LEARNING_RATE, VOCAB_SIZE, MAX_LEN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
train_df = pd.read_csv('../data/processed/train_clean.csv').dropna(subset=['clean_text'])
test_df  = pd.read_csv('../data/processed/test_clean.csv').dropna(subset=['clean_text'])

In [ ]:
# Tokenize with Keras
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['clean_text'])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_df['clean_text']), maxlen=MAX_LEN)
X_test  = pad_sequences(tokenizer.texts_to_sequences(test_df['clean_text']),  maxlen=MAX_LEN)
y_train = train_df['label'].values
y_test  = test_df['label'].values

print('X_train:', X_train.shape, ' X_test:', X_test.shape)

In [ ]:
# DataLoader
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train, dtype=torch.long))
test_ds  = TensorDataset(torch.tensor(X_test,  dtype=torch.long), torch.tensor(y_test,  dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

In [ ]:
# Train
model = LSTMClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}')

In [ ]:
# Evaluate
model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb).argmax(dim=1)
        correct += (preds == yb).sum().item()
        total   += yb.size(0)
print(f'✅ LSTM Accuracy: {correct/total:.4f}')

In [ ]:
# Save
import os
os.makedirs('../saved_models', exist_ok=True)
torch.save(model.state_dict(), '../saved_models/lstm_model.pth')
print('💾 LSTM model saved!')